# MOMAS Taxonomy Experiment Analysis

This notebook visualizes results from the MOMAS taxonomy sweep:
5 settings (team-team, team-individual, team-social, individual-individual, individual-social) across 2 domains (trading, healthcare).

## 1. Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import glob

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

RESULTS_DIR = Path("../results")
DOMAINS = ["trading", "healthcare"]
SETTINGS = ["team-team", "team-individual", "team-social", "individual-individual", "individual-social"]
SETTING_LABELS = {
    "team-team": "Team-Team",
    "team-individual": "Team-Indiv",
    "team-social": "Team-Social",
    "individual-individual": "Indiv-Indiv",
    "individual-social": "Indiv-Social",
}
SETTING_COLORS = {s: c for s, c in zip(SETTINGS, sns.color_palette("husl", len(SETTINGS)))}

print("Setup complete.")

## 2. Load Results

In [ ]:
def load_all_results(results_dir: Path) -> dict:
    """Load all .npz result files into a structured dict.

    Returns:
        dict of form {domain: {setting: [{data dict}, ...]}}
        Each data dict has keys: episode_returns, pareto_front, hypervolume, ser, esr, seed
    """
    results = {}
    npz_files = sorted(glob.glob(str(results_dir / "**" / "*.npz"), recursive=True))

    if not npz_files:
        print("No result files found. Run the experiment sweep first:")
        print("  python -m experiments.sweep")
        return results

    for fpath in npz_files:
        p = Path(fpath)
        # Expected structure: results/{domain}/{setting}/seed_{N}.npz
        try:
            setting = p.parent.name        # e.g. "team-team"
            domain = p.parent.parent.name   # e.g. "trading"
            seed_str = p.stem.replace("seed_", "")  # e.g. "42"
        except Exception:
            print(f"Skipping unrecognized path: {fpath}")
            continue

        data = dict(np.load(fpath, allow_pickle=True))
        data["seed"] = int(seed_str) if seed_str.isdigit() else seed_str

        results.setdefault(domain, {}).setdefault(setting, []).append(data)

    # Print summary
    total = 0
    for domain in sorted(results):
        for setting in sorted(results[domain]):
            n = len(results[domain][setting])
            total += n
            print(f"  {domain}/{setting}: {n} run(s)")
    print(f"\nTotal: {total} result file(s) loaded.")
    return results


results = load_all_results(RESULTS_DIR)
if not results:
    print("\nNotebook cells below will be skipped (no data).")

## 3. Pareto Front Plots

For each domain, plot a 3D scatter of Pareto front points colored by taxonomy setting.

In [ ]:
if results:
    for domain in DOMAINS:
        if domain not in results:
            print(f"No results for domain: {domain}")
            continue

        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection="3d")

        for setting in SETTINGS:
            if setting not in results[domain]:
                continue
            for run in results[domain][setting]:
                pf = run["pareto_front"]
                if pf.ndim < 2 or pf.shape[1] < 3:
                    # Fewer than 3 objectives — plot first 2 or skip
                    continue
                ax.scatter(
                    pf[:, 0], pf[:, 1], pf[:, 2],
                    color=SETTING_COLORS[setting],
                    label=SETTING_LABELS[setting],
                    alpha=0.7,
                    s=30,
                )

        ax.set_xlabel("Objective 1")
        ax.set_ylabel("Objective 2")
        ax.set_zlabel("Objective 3")
        ax.set_title(f"Pareto Front — {domain.title()}")
        # De-duplicate legend entries
        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax.legend(by_label.values(), by_label.keys(), loc="upper left", fontsize=8)
        plt.tight_layout()
        plt.show()
else:
    print("Skipped — no results loaded.")

## 4. Hypervolume Comparison

Bar chart comparing hypervolume across the 5 taxonomy settings for each domain (side-by-side subplots).

In [ ]:
if results:
    present_domains = [d for d in DOMAINS if d in results]
    n_domains = len(present_domains)
    if n_domains == 0:
        print("No domain data available.")
    else:
        fig, axes = plt.subplots(1, n_domains, figsize=(7 * n_domains, 5), sharey=False)
        if n_domains == 1:
            axes = [axes]

        for ax, domain in zip(axes, present_domains):
            means = []
            stds = []
            labels = []
            colors = []
            for setting in SETTINGS:
                if setting not in results[domain]:
                    continue
                hvs = [run["hypervolume"].item() for run in results[domain][setting]]
                means.append(np.mean(hvs))
                stds.append(np.std(hvs) if len(hvs) > 1 else 0.0)
                labels.append(SETTING_LABELS[setting])
                colors.append(SETTING_COLORS[setting])

            x = np.arange(len(labels))
            ax.bar(x, means, yerr=stds, color=colors, edgecolor="black", capsize=4)
            ax.set_xticks(x)
            ax.set_xticklabels(labels, rotation=30, ha="right")
            ax.set_ylabel("Hypervolume")
            ax.set_title(f"Hypervolume — {domain.title()}")

        plt.tight_layout()
        plt.show()
else:
    print("Skipped — no results loaded.")

## 5. SER vs ESR Comparison

Grouped bar chart showing SER and ESR values per taxonomy setting per domain.

In [ ]:
if results:
    present_domains = [d for d in DOMAINS if d in results]
    n_domains = len(present_domains)
    if n_domains == 0:
        print("No domain data available.")
    else:
        fig, axes = plt.subplots(1, n_domains, figsize=(7 * n_domains, 5), sharey=False)
        if n_domains == 1:
            axes = [axes]

        bar_width = 0.35

        for ax, domain in zip(axes, present_domains):
            ser_means = []
            esr_means = []
            labels = []
            for setting in SETTINGS:
                if setting not in results[domain]:
                    continue
                runs = results[domain][setting]
                ser_means.append(np.mean([r["ser"].item() for r in runs]))
                esr_means.append(np.mean([r["esr"].item() for r in runs]))
                labels.append(SETTING_LABELS[setting])

            x = np.arange(len(labels))
            ax.bar(x - bar_width / 2, ser_means, bar_width, label="SER", color="steelblue", edgecolor="black")
            ax.bar(x + bar_width / 2, esr_means, bar_width, label="ESR", color="coral", edgecolor="black")
            ax.set_xticks(x)
            ax.set_xticklabels(labels, rotation=30, ha="right")
            ax.set_ylabel("Value")
            ax.set_title(f"SER vs ESR — {domain.title()}")
            ax.legend()

        plt.tight_layout()
        plt.show()
else:
    print("Skipped — no results loaded.")

## 6. Cross-Domain Comparison

Paired plots showing the same taxonomy setting in trading vs healthcare. Values are min-max normalized within each metric for comparability.

In [ ]:
if results and all(d in results for d in DOMAINS):
    # Gather raw values per (setting, domain) for hypervolume, SER, ESR
    metrics = ["hypervolume", "ser", "esr"]
    raw = {m: {} for m in metrics}  # metric -> {(domain, setting): mean_value}

    for domain in DOMAINS:
        for setting in SETTINGS:
            if setting not in results.get(domain, {}):
                continue
            runs = results[domain][setting]
            for m in metrics:
                raw[m][(domain, setting)] = np.mean([r[m].item() for r in runs])

    # Min-max normalize each metric across all (domain, setting) pairs
    normed = {}
    for m in metrics:
        vals = list(raw[m].values())
        if not vals:
            continue
        lo, hi = min(vals), max(vals)
        rng = hi - lo if hi != lo else 1.0
        normed[m] = {k: (v - lo) / rng for k, v in raw[m].items()}

    # One subplot per taxonomy setting
    common_settings = [s for s in SETTINGS if all((d, s) in raw["hypervolume"] for d in DOMAINS)]
    if not common_settings:
        print("No settings are present in both domains.")
    else:
        n = len(common_settings)
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 5), sharey=True)
        if n == 1:
            axes = [axes]

        bar_width = 0.35
        x = np.arange(len(metrics))

        for ax, setting in zip(axes, common_settings):
            for i, domain in enumerate(DOMAINS):
                vals = [normed[m].get((domain, setting), 0.0) for m in metrics]
                offset = -bar_width / 2 + i * bar_width
                ax.bar(x + offset, vals, bar_width, label=domain.title(), edgecolor="black")

            ax.set_xticks(x)
            ax.set_xticklabels([m.upper() for m in metrics])
            ax.set_title(SETTING_LABELS[setting])
            ax.set_ylim(0, 1.15)
            ax.legend(fontsize=8)

        fig.suptitle("Cross-Domain Comparison (normalized)", fontsize=14, y=1.02)
        plt.tight_layout()
        plt.show()
elif results:
    print("Cross-domain comparison requires both trading and healthcare results.")
else:
    print("Skipped — no results loaded.")

## 7. Summary Table

Plain-text summary of all results (no pandas dependency).

In [ ]:
if results:
    header = f"{'Domain':<14} {'Setting':<24} {'Seeds':>5} {'Hypervolume':>13} {'SER':>10} {'ESR':>10}"
    print(header)
    print("-" * len(header))

    for domain in DOMAINS:
        if domain not in results:
            continue
        for setting in SETTINGS:
            if setting not in results[domain]:
                continue
            runs = results[domain][setting]
            n = len(runs)
            hv = np.mean([r["hypervolume"].item() for r in runs])
            ser = np.mean([r["ser"].item() for r in runs])
            esr = np.mean([r["esr"].item() for r in runs])
            print(f"{domain:<14} {setting:<24} {n:>5} {hv:>13.4f} {ser:>10.4f} {esr:>10.4f}")
        print()  # blank line between domains
else:
    print("No results to summarize. Run the experiment sweep first.")